# FIT DNU CONQUER - Phân tích Dữ liệu Chuỗi Thời gian (Time Series)

## Lab 3: Dự báo Ô nhiễm không khí với SARIMA (Seasonal ARIMA)

**Nhóm:** FIT DNU CONQUER

**Chủ đề 5.3.2:** SARIMA – thêm mùa vụ (seasonality)

**Nội dung thực hiện:**

1.  **Q1: Kiểm tra tính mùa vụ (Seasonality Check):**
    * Sử dụng biểu đồ tự tương quan (ACF) để chứng minh chuỗi dữ liệu có tính mùa vụ (đỉnh ở lag 24, 48...).
2.  **Q2: Xây dựng mô hình SARIMA:**
    * Thiết lập tham số mùa vụ $s=24$ (chu kỳ ngày).
    * Thử nghiệm cấu hình $(p,d,q) \times (P,D,Q,s)$.
3.  **Q3: Đánh giá & So sánh:**
    * So sánh hiệu quả giữa ARIMA (không mùa vụ) và SARIMA (có mùa vụ).
    * Đánh giá qua các chỉ số: AIC, RMSE, MAE.
    * Trực quan hóa kết quả dự báo (Forecast vs Actual).


In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_squared_error, mean_absolute_error

plt.figure(figsize=(15, 6))
plot_acf(df['PM2.5'].dropna(), lags=200)
plt.title("Biểu đồ ACF kiểm tra tính mùa vụ lag 24, 48")
plt.show()

TypeError: 'NoneType' object is not subscriptable

<Figure size 1500x600 with 0 Axes>

In [13]:
p, d, q = 1, 1, 1 
s = 24

best_aic = float("inf")
best_params = None

for P in range(0, 2):
    for D in range(0, 2):
        for Q in range(0, 2):
            try:
                model = SARIMAX(train_data,
                                order=(p, d, q),
                                seasonal_order=(P, D, Q, s),
                                enforce_stationarity=False,
                                enforce_invertibility=False)
                results = model.fit(disp=False)
                if results.aic < best_aic:
                    best_aic = results.aic
                    best_params = (P, D, Q)
                    sarima_final = results
            except:
                continue

print("Cấu hình SARIMA tối ưu:", best_params)

Cấu hình SARIMA tối ưu: None


In [14]:
arima_model = SARIMAX(train_data, order=(p, d, q)).fit(disp=False)
arima_pred = arima_model.forecast(steps=len(test_data))
sarima_pred = sarima_final.forecast(steps=len(test_data))

def calc_metrics(actual, pred):
    rmse = np.sqrt(mean_squared_error(actual, pred))
    mae = mean_absolute_error(actual, pred)
    return rmse, mae

rmse_ari, mae_ari = calc_metrics(test_data, arima_pred)
rmse_sar, mae_sar = calc_metrics(test_data, sarima_pred)

print("Bảng so sánh kết quả:")
data_compare = {
    "Chỉ số": ["AIC", "RMSE", "MAE"],
    "ARIMA": [arima_model.aic, rmse_ari, mae_ari],
    "SARIMA": [sarima_final.aic, rmse_sar, mae_sar]
}
print(pd.DataFrame(data_compare))

NameError: name 'train_data' is not defined

In [15]:
plt.figure(figsize=(15, 7))
plt.plot(test_data.index, test_data, label='Actual')
plt.plot(test_data.index, arima_pred, label='ARIMA', linestyle='--')
plt.plot(test_data.index, sarima_pred, label='SARIMA', color='red')
plt.legend()
plt.title("So sánh dự báo ARIMA và SARIMA với thực tế")
plt.show()

NameError: name 'test_data' is not defined

<Figure size 1500x700 with 0 Axes>